# pycatdap tutorial — AIC-based categorical data analysis

**CATDAP** (CATegorical Data Analysis Program) applies Akaike's Information Criterion (AIC) to categorical data analysis. It was originally developed in 1980 by Y. Sakamoto and M. Katsura at the Institute of Statistical Mathematics, Japan.

`pycatdap` is a Python implementation of the R `catdap` package, exposing the two main routines:

| Function | Purpose |
|------|------|
| `catdap1()` | Evaluate the pairwise association of every variable pair via ΔAIC |
| `catdap2()` | Search for the optimal explanatory subset, including automatic categorization of continuous variables |

## Notebook outline

1. **Mathematical foundation** — AIC definition and interpretation
2. **Data exploration** — inspect the bundled dataset
3. **CATDAP-01** — pairwise association analysis with visualization
4. **CATDAP-02** — optimal subset search (with pooling) and visualization
5. **Large-scale data** — execution on the HelloGoodbye dataset
6. **Sanity checks** — structural and numerical consistency of the results


In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import pycatdap
from pycatdap.datasets import load_health_data, load_hello_goodbye
from pycatdap.plotting import aic_comparison_plot, barplot_twoway, mosaic_plot

%matplotlib inline
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["font.size"] = 11
warnings.filterwarnings("ignore", category=FutureWarning)

print(f"pycatdap version: {pycatdap.__version__}")

## 1. Mathematical foundation

### AIC for a contingency-table model

For a response variable $E$ and an explanatory variable $F$:

$$AIC(E; F) = -2 \sum_{i,j} n_{EF}(i,j) \ln \frac{n_{EF}(i,j)}{n_F(j)} + 2(C_E - 1) C_F$$

- $n_{EF}(i,j)$: cross frequency (response category $i$, explanatory category $j$)
- $n_F(j)$: marginal frequency of the explanatory variable
- $C_E, C_F$: number of categories of each variable

### Base AIC (null model with no explanatory variable)

$$AIC(E; \phi) = -2 \sum_i n_E(i) \ln \frac{n_E(i)}{n} + 2(C_E - 1)$$

### ΔAIC (the reported value)

$$\Delta AIC = AIC(E; F) - AIC(E; \phi)$$

| ΔAIC value | Interpretation |
|-----------|------|
| $\Delta AIC < 0$ | Explanatory variable $F$ is **informative** for $E$ |
| $\Delta AIC \geq 0$ | Explanatory variable $F$ is **uninformative** (the penalty outweighs the fit gain) |

> **Note**: zero-frequency cells follow the convention $0 \times \ln(0) = 0$.


## 2. Data exploration

### HealthData (52 observations, 8 variables)

A medical dataset bundled with the R `catdap` package. It mixes categorical and continuous variables.

| Variable | Kind | Description |
|------|-----|------|
| opthalmo. | categorical (1, 2) | Ophthalmic examination result |
| ecg | categorical (1, 2) | Electrocardiogram result |
| **symptoms** | categorical (A, B) | **Symptom classification (response variable)** |
| age | continuous | Age |
| max.press | continuous | Maximum blood pressure |
| min.press | continuous | Minimum blood pressure |
| aortic.wav | continuous | Aortic wave measurement |
| cholesterol | categorical (low, high) | Cholesterol level |


In [ ]:
df = load_health_data()
print(f"Shape: {df.shape}")
df.head(10)

In [ ]:
df.describe(include="all").round(2)

## 3. CATDAP-01 — pairwise association analysis

`catdap1()` operates on **categorical variables only** and computes the ΔAIC for every variable pair.

- `response_names=None` (default): treat every column as a response in turn
- `response_names=["symptoms"]`: restrict the analysis to the listed response variables

> **Note**: if you pass a continuous variable directly, every unique value becomes its own category and the penalty term explodes. Use the pooling feature of `catdap2()` for continuous variables.


In [ ]:
# Extract the categorical columns and run catdap1
cat_df = df[["symptoms", "opthalmo.", "ecg", "cholesterol"]]

result1 = pycatdap.catdap1(cat_df)
print(result1)
print("\n--- ΔAIC matrix ---")
result1.aic

In [ ]:
# Ranking of explanatory variables for `symptoms`
print("Explanatory ranking for symptoms (ΔAIC ascending = best first):")
for i, var in enumerate(result1.aic_order["symptoms"], 1):
    aic_val = result1.aic.loc["symptoms", var]
    marker = "OK" if aic_val < 0 else "--"
    print(f"  {i}. {var:15s}  ΔAIC = {aic_val:+.4f}  {marker}")

### Visualization

The ΔAIC bar chart (green = informative, red = uninformative) alongside the two-way table for the top-ranked explanatory variable.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# ΔAIC comparison bar chart
aic_comparison_plot(result1, response="symptoms", ax=axes[0])

# Stacked bar chart of the best two-way table
best_var = result1.aic_order["symptoms"][0]
tway = result1.tway_tables[("symptoms", best_var)]
barplot_twoway(tway, ax=axes[1])
axes[1].set_title(f"symptoms × {best_var}")

# Mosaic plot
mosaic_plot(tway, ax=axes[2])
axes[2].set_title(f"Mosaic: symptoms × {best_var}")

plt.tight_layout()
plt.show()

### ΔAIC is asymmetric

ΔAIC is not symmetric: $\Delta AIC(\text{symptoms} \to \text{ecg}) \neq \Delta AIC(\text{ecg} \to \text{symptoms})$.


In [ ]:
aic_s_e = result1.aic.loc["symptoms", "ecg"]
aic_e_s = result1.aic.loc["ecg", "symptoms"]
print(f"ΔAIC(symptoms -> ecg)   = {aic_s_e:+.4f}")
print(f"ΔAIC(ecg -> symptoms)   = {aic_e_s:+.4f}")
print(f"diff = {abs(aic_s_e - aic_e_s):.4f}  -> asymmetry confirmed")

## 4. CATDAP-02 — optimal explanatory subset search

`catdap2()` extends `catdap1()` with two extra capabilities:

1. **Pooling for continuous variables** — automatically split a continuous variable into categories that minimize the AIC.
2. **Multi-variable subset search** — incrementally consider 1, 2, ... variable combinations and report the best subset for each size.

### `pool` parameter

| Value | Meaning | Target |
|----|------|------|
| `0` | Equal-width pooling (top-down) | Continuous variables |
| `1` | Unequal pooling (bottom-up, default) | Continuous variables |
| `2` | No pooling | Categorical variables |

HealthData configuration:
```
pool = [2, 2, 2, 0, 0, 0, 0, 2]
       opthalmo. ecg symptoms age max.press min.press aortic.wav cholesterol
       ^cat     ^cat ^cat    ^cont ^cont   ^cont    ^cont      ^cat
```


In [ ]:
result2 = pycatdap.catdap2(
    df,
    pool=[2, 2, 2, 0, 0, 0, 0, 2],
    response_name="symptoms",
    accuracy=[0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 0.1, 0.0],
)
print(result2)

In [ ]:
# Base AIC (null model with no explanatory variable)
print(f"Base AIC (null model): {result2.base_aic:.4f}")
print("\n--- Single-variable ΔAIC ranking ---")
result2.aic

In [ ]:
# ΔAIC comparison bar chart
fig, ax = plt.subplots(figsize=(8, 4))
aic_comparison_plot(result2, ax=ax)
plt.tight_layout()
plt.show()

### Pooling result

Inspect how each continuous variable was categorized.


In [ ]:
print("--- Pooling boundaries ---")
for var, boundaries in result2.intervals.items():
    n_bins = len(boundaries) + 1
    print(f"\n  {var}: {n_bins} bins")
    if boundaries:
        print(f"    boundaries: {[round(b, 2) for b in boundaries]}")
    else:
        print("    (single bin — no split)")

### Best subsets

The best combination of explanatory variables for each subset size, with its ΔAIC.


In [ ]:
print("--- Best subsets ---")
seen_nvars = set()
for s in result2.subsets:
    if s.n_vars not in seen_nvars:
        seen_nvars.add(s.n_vars)
        vars_str = ", ".join(s.variables)
        aic_str = f"ΔAIC = {s.aic:+.4f}"
        print(f"  {s.n_vars} vars: [{vars_str}]  {aic_str}")

## 5. Large-scale data — HelloGoodbye

HelloGoodbye is a 13,954-row dataset with 56 binary variables. The `nvar` parameter restricts the search to the top-`k` candidates, keeping computation tractable on wide tables.


In [ ]:
df_hg = load_hello_goodbye()
print(f"Shape: {df_hg.shape}")
print(f"Response: Isay — values: {sorted(df_hg['Isay'].unique())}")
print(f"All binary: {all(set(df_hg[c].unique()) <= {0, 1} for c in df_hg.columns)}")
df_hg.head()

In [ ]:
%%time
# All variables are binary, so pool=2 everywhere; nvar=5 limits the search
pool_hg = [2] * len(df_hg.columns)
result_hg = pycatdap.catdap2(df_hg, pool=pool_hg, response_name="Isay", nvar=5)
print(result_hg)

In [ ]:
# Top-10 ΔAIC values
print("--- Top 10 variables for Isay ---")
result_hg.aic.head(10)

In [ ]:
# Top-15 bar chart
top15 = result_hg.aic.head(15)
fig, ax = plt.subplots(figsize=(8, 5))
colors = ["tab:green" if v < 0 else "tab:red" for v in top15["aic"]]
ax.barh(top15["variable"], top15["aic"], color=colors)
ax.set_xlabel("ΔAIC")
ax.set_title("HelloGoodbye: Top 15 variables for Isay")
ax.axvline(0, color="black", linewidth=0.8)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 6. Sanity checks

The checks below confirm that pycatdap's output is mathematically consistent.

For a strict numerical comparison against the R `catdap` package, run `docs/r_reference/generate_reference.R` and diff its CSV output against the Python output. The checks here are **structural and mathematical consistency** that do not require an R installation.


In [ ]:
from pycatdap._aic import (
    _safe_xlogy,
    compute_aic_twoway,
    compute_base_aic,
    compute_delta_aic,
)
from pycatdap._contingency import build_crosstab

checks = []

# --- Check 1: ΔAIC = AIC(E;F) - AIC(E;φ) identity ---
cross, marg_e, marg_f, n = build_crosstab(cat_df, "symptoms", "cholesterol")
delta = compute_delta_aic(cross, marg_e, marg_f, n)
twoway = compute_aic_twoway(cross, marg_f)
base = compute_base_aic(marg_e, n)
ok1 = np.isclose(delta, twoway - base, rtol=1e-10)
checks.append(("ΔAIC = AIC(E;F) - AIC(E;phi) identity", ok1))

# --- Check 2: independent variables -> ΔAIC ~ 0 (non-negative) ---
rng = np.random.default_rng(42)
ind_df = pd.DataFrame(
    {
        "Y": rng.choice(["a", "b"], size=10000),
        "X": rng.choice(["p", "q", "r"], size=10000),
    }
)
r_ind = pycatdap.catdap1(ind_df, response_names=["Y"])
ok2 = r_ind.aic.loc["Y", "X"] > -1.0
checks.append(("Independent variables -> ΔAIC ~ 0 (non-negative)", bool(ok2)))

# --- Check 3: perfect association -> ΔAIC << 0 ---
perf_df = pd.DataFrame(
    {
        "Y": ["a"] * 50 + ["b"] * 50,
        "X": ["p"] * 50 + ["q"] * 50,
    }
)
r_perf = pycatdap.catdap1(perf_df, response_names=["Y"])
ok3 = r_perf.aic.loc["Y", "X"] < -10.0
checks.append(("Perfect association -> ΔAIC << 0", bool(ok3)))

# --- Check 4: base AIC > 0 ---
ok4 = result2.base_aic > 0
checks.append(("Base AIC > 0 on a non-trivial dataset", bool(ok4)))

# --- Check 5: all ΔAIC values are finite ---
ok5 = bool(np.all(np.isfinite(result2.aic["aic"].to_numpy())))
checks.append(("All ΔAIC values are finite (no NaN/Inf)", ok5))

# --- Check 6: aortic.wav is the best single-variable predictor in catdap2 ---
ok6 = result2.aic_order[0] == "aortic.wav"
checks.append(("catdap2: aortic.wav is the best single predictor", bool(ok6)))

# --- Check 7: pooling boundaries exist for continuous variables ---
cont_vars = ["age", "max.press", "min.press", "aortic.wav"]
ok7 = all(v in result2.intervals for v in cont_vars)
checks.append(("Pooling boundaries exist for continuous variables", ok7))

# --- Check 8: subset search reaches subsets of size >= 2 ---
ok8 = max(s.n_vars for s in result2.subsets) >= 2
checks.append(("Subset search reaches subsets of size >= 2", bool(ok8)))

# --- Check 9: catdap1 ΔAIC is asymmetric ---
ok9 = result1.aic.loc["symptoms", "ecg"] != result1.aic.loc["ecg", "symptoms"]
checks.append(("catdap1 ΔAIC is asymmetric", bool(ok9)))

# --- Check 10: 0 * ln(0) = 0 convention ---
ok10 = float(_safe_xlogy(np.array([0.0]), np.array([0.0]))[0]) == 0.0
checks.append(("0 * ln(0) = 0 convention", ok10))

# --- Report ---
print("=" * 55)
print("Sanity check summary")
print("=" * 55)
all_ok = True
for name, passed in checks:
    status = "PASS" if passed else "FAIL"
    print(f"  [{status}]  {name}")
    if not passed:
        all_ok = False
print("=" * 55)
if all_ok:
    print("All 10 checks PASS — results are consistent")
else:
    print("WARNING: one or more checks failed")

## References

- Akaike, H. (1974). "A new look at the statistical model identification." *IEEE Transactions on Automatic Control*, 19(6), 716-723.
- Sakamoto, Y. and Katsura, M. (1980). "Analysis program for categorical data: CATDAP." Institute of Statistical Mathematics, Japan.
- R `catdap` package: https://cran.r-project.org/package=catdap

---

### Strict numerical comparison against the R reference

If R is available, run `docs/r_reference/generate_reference.R` to produce the CSV files used as the ground truth for the comparison tests:

```r
Rscript docs/r_reference/generate_reference.R
```

Generated CSV files:
- `health_catdap1.csv` — `catdap1` ΔAIC matrix (categorical columns)
- `health_catdap2_aic.csv` — `catdap2` single-variable ΔAIC
- `health_catdap2_subsets.csv` — `catdap2` best-subset table
